# CS224N 作业一：探索词向量（25分）
### <font color='blue'> 截止时间：2024年4月9日（周二）下午4:30</font>

欢迎来到 CS224N！

在开始之前，请确保**阅读与本笔记本同一目录下的 README.md**，以获取重要的环境配置信息。在成功完成本作业之前，您需要安装一些 Python 库。本笔记本中提供了大量代码，我们强烈建议您将其作为学习的一部分来阅读和理解 :)

如果您对 Python、Numpy 或 Matplotlib 不太熟悉，建议您参加周五的复习课程。该课程将被录制，相关材料将在我们的[网站](http://web.stanford.edu/class/cs224n/index.html#schedule)上提供。CS231N 的 Python/Numpy [教程](https://cs231n.github.io/python-numpy-tutorial/)也是一个很好的学习资源。

**作业须知：** 请务必在完成过程中随时保存笔记本。提交说明位于笔记本的底部。

In [ ]:
# 所有导入语句均在此定义
# 注意：请勿向此列表中添加内容。
# ----------------

import sys
assert sys.version_info[0] == 3
assert sys.version_info[1] >= 8

from platform import python_version
assert int(python_version().split(".")[1]) >= 5, "请按照与本笔记本同一目录下的 README.md 文件中的说明升级您的 Python 版本。您的 Python 版本为 " + python_version()

from gensim.models import KeyedVectors
from gensim.test.utils import datapath
import pprint
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = [10, 5]

from datasets import load_dataset
imdb_dataset = load_dataset("stanfordnlp/imdb")

import re
import numpy as np
import random
import scipy as sp
from sklearn.decomposition import TruncatedSVD
from sklearn.decomposition import PCA

START_TOKEN = '<START>'
END_TOKEN = '<END>'
NUM_SAMPLES = 150

np.random.seed(0)
random.seed(0)
# ----------------

## 词向量

词向量通常作为下游 NLP 任务（如问答、文本生成、翻译等）的基础组件，因此建立对其优势和不足的直觉认识非常重要。在此，您将探索两种类型的词向量：一种源自*共现矩阵*，另一种源自 *GloVe*。

**术语说明：** "词向量"（word vectors）和"词嵌入"（word embeddings）这两个术语经常交替使用。"嵌入"一词指的是我们将词语的语义信息编码到更低维的空间中。正如[维基百科](https://en.wikipedia.org/wiki/Word_embedding)所述，"*从概念上讲，它涉及从每个词对应一个维度的空间到维度低得多的连续向量空间的数学映射*"。

## 第一部分：基于计数的词向量（10分）

大多数词向量模型都始于以下理念：

*您可以通过一个词的伴随词来认识这个词（[Firth, J. R. 1957:11](https://en.wikipedia.org/wiki/John_Rupert_Firth)）*

许多词向量实现的驱动力在于这样一个想法：相似的词，即（近）同义词，会在相似的上下文中使用。因此，相似的词往往会与一组共同的词（即上下文）一起被说出或写下。通过考察这些上下文，我们可以尝试为词语构建嵌入。基于这一直觉，许多"传统"的词向量构建方法依赖于词频统计。在此，我们将详细阐述其中一种策略——*共现矩阵*（更多信息请参见[此处](https://web.stanford.edu/~jurafsky/slp3/6.pdf)或[此处](https://web.archive.org/web/20190530091127/https://medium.com/data-science-group-iitr/word-embedding-2d05d270b285)）。

### 共现(Co-Occurrence)

共现矩阵用于统计在某一环境中事物共同出现的频率。给定某个词 $w_i$ 在文档中出现，我们考察其周围的"上下文窗口"（context window）。假设固定的窗口大小为 $n$ ，则该窗口包含 $w_i$ 前面的 $n$ 个词和后面的 $n$ 个词，即词序列 $w_{i-n} \dots w_{i-1}$ 和 $w_{i+1} \dots w_{i+n}$ 。 我们构建一个共现矩阵 $M$ ，它是一个对称的词-词矩阵，其中元素 $M_{ij}$ 表示：在所有文档中，词 $w_j$ 出现在词 $w_i$ 的窗口内的总次数。

**示例：固定窗口 n=1 的共现矩阵**：

文档 1: "all that glitters is not gold"

文档 2: "all is well that ends well"


|     *    | `<START>` | all | that | glitters | is   | not  | gold  | well | ends | `<END>` |
|----------|-------|-----|------|----------|------|------|-------|------|------|-----|
| `<START>`    | 0     | 2   | 0    | 0        | 0    | 0    | 0     | 0    | 0    | 0   |
| all      | 2     | 0   | 1    | 0        | 1    | 0    | 0     | 0    | 0    | 0   |
| that     | 0     | 1   | 0    | 1        | 0    | 0    | 0     | 1    | 1    | 0   |
| glitters | 0     | 0   | 1    | 0        | 1    | 0    | 0     | 0    | 0    | 0   |
| is       | 0     | 1   | 0    | 1        | 0    | 1    | 0     | 1    | 0    | 0   |
| not      | 0     | 0   | 0    | 0        | 1    | 0    | 1     | 0    | 0    | 0   |
| gold     | 0     | 0   | 0    | 0        | 0    | 1    | 0     | 0    | 0    | 1   |
| well     | 0     | 0   | 1    | 0        | 1    | 0    | 0     | 0    | 1    | 1   |
| ends     | 0     | 0   | 1    | 0        | 0    | 0    | 0     | 1    | 0    | 0   |
| `<END>`      | 0     | 0   | 0    | 0        | 0    | 0    | 1     | 1    | 0    | 0   |

在 NLP 中，我们通常使用 `<START>` 和 `<END>` 标记来标示句子、段落或文档的开头和结尾。这些标记被纳入共现统计中，将每个文档包裹起来，例如："`<START>` All that glitters is not gold `<END>`"。

矩阵的行（或列）提供了基于词-词共现的词向量，但其规模可能很大。为了降低维度，我们采用奇异值分解（SVD），类似于主成分分析（PCA），选取前 $k$ 个主成分。SVD 过程将共现矩阵 $A$ 分解为对角矩阵 $S$ 中的奇异值和 $U_k$ 中的新的、更短的词向量。

这种降维操作能够保持语义关系；例如，*doctor*（医生）和 *hospital*（医院）之间的距离会比 *doctor* 和 *dog*（狗）更近。

对于不熟悉特征值和 SVD 的读者，[此处](https://davetang.org/file/Singular_Value_Decomposition_Tutorial.pdf)提供了一篇适合初学者的 SVD 入门教程。其他深入学习的资源包括 CS168 课程的第 [7](https://web.stanford.edu/class/cs168/l/l7.pdf)、[8](http://theory.stanford.edu/~tim/s15/l/l8.pdf)、[9](https://web.stanford.edu/class/cs168/l/l9.pdf) 讲，对这些算法进行了较高层次的讲解。在实际实现中，建议使用 numpy、scipy 或 sklearn 等 Python 包中预编程的函数。虽然对大型语料库应用完整 SVD 可能会消耗大量内存，但截断 SVD（Truncated SVD）等可扩展技术可以高效地提取前 $k$ 个向量分量。

### 绘制共现词嵌入

在此，我们将使用大型电影评论数据集（Large Movie Review Dataset）。这是一个用于二元情感分类的数据集，包含比先前基准数据集更多的数据。我们提供了 25,000 条高度极化的电影评论用于训练，以及 25,000 条用于测试。此外还有额外的未标注数据可供使用。我们在下方提供了 `read_corpus` 函数，用于从数据集中提取电影评论文本。该函数还会在每个文档中添加 `<START>` 和 `<END>` 标记，并将单词转换为小写。您**无需**执行任何其他类型的预处理。

In [ ]:
def read_corpus():
    """ 从大型电影评论数据集中读取文件。
        参数：
            category (字符串): 类别名称
        返回：
            列表的列表，包含每个已处理文件中的单词
    """
    files = imdb_dataset["train"]["text"][:NUM_SAMPLES]
    return [[START_TOKEN] + [re.sub(r'[^\w]', '', w.lower()) for w in f.split(" ")] + [END_TOKEN] for f in files]


让我们看看这些文档是什么样子的……

In [ ]:
imdb_corpus = read_corpus()
pprint.pprint(imdb_corpus[:3], compact=True, width=100)
print("corpus size: ", len(imdb_corpus[0]))

### 问题 1.1：实现 `distinct_words` [编程]（2分）

编写一个方法，计算语料库中出现的不同单词（词类型）。

您可以使用 `for` 循环来处理输入的 `corpus`（字符串列表的列表），但建议尝试使用 Python 列表推导式（通常速度更快）。具体来说，[这篇文档](https://coderwall.com/p/rcmaea/flatten-a-list-of-lists-in-one-line-in-python)可能有助于展平列表的列表。如果您对 Python 列表推导式不太熟悉，可以参考[更多信息](https://python-3-patterns-idioms-test.readthedocs.io/en/latest/Comprehensions.html)。

返回的 `corpus_words` 应当是已排序的。您可以使用 Python 的 `sorted` 函数来实现。

使用 [Python 集合](https://www.w3schools.com/python/python_sets.asp)来去除重复单词可能会有所帮助。

In [ ]:
def distinct_words(corpus):
    """ 确定语料库中不同单词的列表。
        参数：
            corpus（字符串列表的列表）：文档语料库
        返回：
            corpus_words（字符串列表）：整个语料库中不同单词的排序列表
            n_corpus_words（整数）：整个语料库中不同单词的数量
    """
    corpus_words = []
    n_corpus_words = -1
    
    # ------------------
    # 在此编写您的实现代码。
    
    
    # ------------------

    return corpus_words, n_corpus_words

In [ ]:
# ---------------------
# 运行此健全性检查
# 注意：这不是对正确性的详尽检查。
# ---------------------

# 定义测试语料库
test_corpus = ["{} All that glitters isn't gold {}".format(START_TOKEN, END_TOKEN).split(" "), "{} All's well that ends well {}".format(START_TOKEN, END_TOKEN).split(" ")]
test_corpus_words, num_corpus_words = distinct_words(test_corpus)

# 正确答案
ans_test_corpus_words = sorted([START_TOKEN, "All", "ends", "that", "gold", "All's", "glitters", "isn't", "well", END_TOKEN])
ans_num_corpus_words = len(ans_test_corpus_words)

# 测试单词数量是否正确
assert(num_corpus_words == ans_num_corpus_words), "不同单词的数量不正确。正确答案：{}。您的答案：{}".format(ans_num_corpus_words, num_corpus_words)

# 测试单词是否正确
assert (test_corpus_words == ans_test_corpus_words), "corpus_words 不正确。\n正确答案：{}\n您的答案：  {}".format(str(ans_test_corpus_words), str(test_corpus_words))

# 打印成功信息
print ("-" * 80)
print("通过所有测试！")
print ("-" * 80)

### 问题 1.2：实现 `compute_co_occurrence_matrix` [编程]（3分）

编写一个方法，为特定的窗口大小 $n$（默认为 4）构建共现矩阵，考虑窗口中心词前 $n$ 个和后 $n$ 个词。在此，我们开始使用 `numpy (np)` 来表示向量、矩阵和张量。如果您对 NumPy 不熟悉，cs231n 的 [Python NumPy 教程](http://cs231n.github.io/python-numpy-tutorial/)后半部分提供了 NumPy 教程。


In [ ]:
def compute_co_occurrence_matrix(corpus, window_size=4):
    """ 计算给定语料库和窗口大小（默认为 4）的共现矩阵。
    
        注意：文档中的每个词都应位于窗口的中心。靠近边缘的词将具有较少的共现词。
              
              例如，对于文档 "<START> All that glitters is not gold <END>"，窗口大小为 4 时，
              "All" 将与 "<START>"、"that"、"glitters"、"is" 和 "not" 共现。
    
        参数：
            corpus（字符串列表的列表）：文档语料库
            window_size（整数）：上下文窗口大小
        返回：
            M（形状为 (语料库中不同单词数, 语料库中不同单词数) 的对称 numpy 矩阵）：
                词频共现矩阵。
                行/列中单词的顺序应与 distinct_words 函数给出的单词顺序一致。
            word2ind（字典）：将单词映射到索引（即矩阵 M 的行/列号）的字典。
    """
    words, n_words = distinct_words(corpus)
    M = None
    word2ind = {}
    
    # ------------------
    # 在此编写您的实现代码。
    
    
    # ------------------

    return M, word2ind

In [ ]:
# ---------------------
# 运行此健全性检查
# 注意：这不是对正确性的详尽检查。
# ---------------------

# 定义测试语料库并获取学生的共现矩阵
test_corpus = ["{} All that glitters isn't gold {}".format(START_TOKEN, END_TOKEN).split(" "), "{} All's well that ends well {}".format(START_TOKEN, END_TOKEN).split(" ")]
M_test, word2ind_test = compute_co_occurrence_matrix(test_corpus, window_size=1)

# 正确的 M 和 word2ind
M_test_ans = np.array( 
    [[0., 0., 0., 0., 0., 0., 1., 0., 0., 1.,],
     [0., 0., 1., 1., 0., 0., 0., 0., 0., 0.,],
     [0., 1., 0., 0., 0., 0., 0., 0., 1., 0.,],
     [0., 1., 0., 0., 0., 0., 0., 0., 0., 1.,],
     [0., 0., 0., 0., 0., 0., 0., 0., 1., 1.,],
     [0., 0., 0., 0., 0., 0., 0., 1., 1., 0.,],
     [1., 0., 0., 0., 0., 0., 0., 1., 0., 0.,],
     [0., 0., 0., 0., 0., 1., 1., 0., 0., 0.,],
     [0., 0., 1., 0., 1., 1., 0., 0., 0., 1.,],
     [1., 0., 0., 1., 1., 0., 0., 0., 1., 0.,]]
)
ans_test_corpus_words = sorted([START_TOKEN, "All", "ends", "that", "gold", "All's", "glitters", "isn't", "well", END_TOKEN])
word2ind_ans = dict(zip(ans_test_corpus_words, range(len(ans_test_corpus_words))))

# 测试 word2ind 是否正确
assert (word2ind_ans == word2ind_test), "您的 word2ind 不正确：\n正确答案：{}\n您的答案：{}".format(word2ind_ans, word2ind_test)

# 测试 M 的形状是否正确
assert (M_test.shape == M_test_ans.shape), "M 矩阵的形状不正确。\n正确答案：{}\n您的答案：{}".format(M_test.shape, M_test_ans.shape)

# 测试 M 的值是否正确
for w1 in word2ind_ans.keys():
    idx1 = word2ind_ans[w1]
    for w2 in word2ind_ans.keys():
        idx2 = word2ind_ans[w2]
        student = M_test[idx1, idx2]
        correct = M_test_ans[idx1, idx2]
        if student != correct:
            print("正确的 M：")
            print(M_test_ans)
            print("您的 M：")
            print(M_test)
            raise AssertionError("矩阵 M 中索引 ({}, {})=({}, {}) 处的计数值不正确。您的值为 {}，但应为 {}。".format(idx1, idx2, w1, w2, student, correct))

# 打印成功信息
print ("-" * 80)
print("通过所有测试！")
print ("-" * 80)

### 问题 1.3：实现 `reduce_to_k_dim` [编程]（1分）

构建一个方法，对矩阵进行降维以生成 k 维嵌入。使用 SVD 提取前 k 个分量，生成一个新的 k 维嵌入矩阵。

**注意：** numpy、scipy 和 scikit-learn（`sklearn`）都提供了 SVD 的*某种*实现，但只有 scipy 和 sklearn 提供了截断 SVD（Truncated SVD）的实现，且只有 sklearn 提供了用于计算大规模截断 SVD 的高效随机算法。因此请使用 [sklearn.decomposition.TruncatedSVD](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html)。

In [ ]:
def reduce_to_k_dim(M, k=2):
    """ 使用 Scikit-Learn 的以下 SVD 函数，将维度为 (语料库单词数, 语料库单词数) 的共现计数矩阵
        降维为维度为 (语料库单词数, k) 的矩阵：
            - http://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html
    
        参数：
            M（形状为 (语料库中不同单词数, 语料库中不同单词数) 的 numpy 矩阵）：词频共现矩阵
            k（整数）：降维后每个词的嵌入维度
        返回：
            M_reduced（形状为 (语料库单词数, k) 的 numpy 矩阵）：k 维词嵌入矩阵。
                    就数学课上的 SVD 而言，这实际上返回的是 U * S
    """    
    n_iters = 10    # 在调用 `TruncatedSVD` 时使用此参数
    M_reduced = None
    print("正在对 %i 个词运行截断 SVD..." % (M.shape[0]))
    
    # ------------------
    # 在此编写您的实现代码。
    
    
    # ------------------

    print("完成。")
    return M_reduced

In [ ]:
# ---------------------
# 运行此健全性检查
# 注意：这不是对正确性的详尽检查。
# 实际上我们仅检查您的 M_reduced 是否具有正确的维度。
# ---------------------

# 定义测试语料库并运行学生代码
test_corpus = ["{} All that glitters isn't gold {}".format(START_TOKEN, END_TOKEN).split(" "), "{} All's well that ends well {}".format(START_TOKEN, END_TOKEN).split(" ")]
M_test, word2ind_test = compute_co_occurrence_matrix(test_corpus, window_size=1)
M_test_reduced = reduce_to_k_dim(M_test, k=2)

# 测试维度是否正确
assert (M_test_reduced.shape[0] == 10), "M_reduced 有 {} 行；应有 {}".format(M_test_reduced.shape[0], 10)
assert (M_test_reduced.shape[1] == 2), "M_reduced 有 {} 列；应有 {}".format(M_test_reduced.shape[1], 2)

# 打印成功信息
print ("-" * 80)
print("通过所有测试！")
print ("-" * 80)

### 问题 1.4：实现 `plot_embeddings` [编程]（1分）

在此，您将编写一个函数，在二维空间中绘制一组二维向量。绘图方面，我们将使用 Matplotlib（`plt`）。

对于本例，您可能会发现参考[这段代码](http://web.archive.org/web/20190924160434/https://www.pythonmembers.club/2018/05/08/matplotlib-scatter-plot-annotate-set-text-at-label-each-point/)很有用。将来，制作图表的好方法是查看 [Matplotlib 画廊](https://matplotlib.org/gallery/index.html)，找到一个与您需求相似的图表，并改编其提供的代码。

In [ ]:
def plot_embeddings(M_reduced, word2ind, words):
    """ 在散点图中绘制列表 "words" 中指定单词的嵌入。
        注意：不要绘制 M_reduced / word2ind 中列出的所有单词。
        在每个点旁边添加标签。
        
        参数：
            M_reduced（形状为 (语料库中不同单词数, 2) 的 numpy 矩阵）：二维词嵌入矩阵
            word2ind（字典）：将单词映射到矩阵 M 索引的字典
            words（字符串列表）：需要可视化的嵌入所对应的单词
    """

    # ------------------
    # 在此编写您的实现代码。
    
    
    # ------------------

In [ ]:
# ---------------------
# 运行此健全性检查
# 注意：这不是对正确性的详尽检查。
# 生成的图表应类似于附带的 question_1.4_test.png 文件。
# ---------------------

print ("-" * 80)
print ("输出的图表：")

M_reduced_plot_test = np.array([[1, 1], [-1, -1], [1, -1], [-1, 1], [0, 0]])
word2ind_plot_test = {'test1': 0, 'test2': 1, 'test3': 2, 'test4': 3, 'test5': 4}
words = ['test1', 'test2', 'test3', 'test4', 'test5']
plot_embeddings(M_reduced_plot_test, word2ind_plot_test, words)

print ("-" * 80)

### 问题 1.5：共现绘图分析 [文字题]（3分）

现在我们将把您编写的所有部分整合在一起！我们将基于大型电影评论语料库，以固定窗口大小 4（默认窗口大小）计算共现矩阵。然后使用 TruncatedSVD 计算每个词的二维嵌入。TruncatedSVD 返回的是 U\*S，因此我们需要对返回的向量进行归一化，使所有向量出现在单位圆附近（因此 closeness 表示方向上的接近度）。**注意**：下面进行归一化的代码行使用了 NumPy 的*广播*（broadcasting）概念。如果您不了解广播，请参阅 [Jake VanderPlas 的《数组计算：广播》](https://jakevdp.github.io/PythonDataScienceHandbook/02.05-computation-on-arrays-broadcasting.html)。

运行以下单元格以生成图表。运行可能需要几分钟时间。

In [ ]:
# -----------------------------
# 运行此单元格以生成您的图表
# ------------------------------
imdb_corpus = read_corpus()
M_co_occurrence, word2ind_co_occurrence = compute_co_occurrence_matrix(imdb_corpus)
M_reduced_co_occurrence = reduce_to_k_dim(M_co_occurrence, k=2)

# 重新缩放（归一化）各行，使每行具有单位长度
M_lengths = np.linalg.norm(M_reduced_co_occurrence, axis=1)
M_normalized = M_reduced_co_occurrence / M_lengths[:, np.newaxis] # 广播

words = ['movie', 'book', 'mysterious', 'story', 'fascinating', 'good', 'interesting', 'large', 'massive', 'huge']

plot_embeddings(M_normalized, word2ind_co_occurrence, words)

**请验证您的图与作业压缩包中的 "question_1.5.png" 一致。如果不一致，请使用 "question_1.5.png" 中的图来回答接下来的两个问题。**

a. 找出至少两组在二维嵌入空间中聚在一起的单词。对您观察到的每个聚类给出解释。

#### <font color="red">在此处填写您的答案。</font>


b. 有哪些您认为应该聚在一起但实际上没有的单词？请描述至少两个例子。

#### <font color="red">在此处填写您的答案。</font>

## 第二部分：基于预测的词向量（15分）

如课堂所述，近年来基于预测的词向量展现出了更好的性能，如 word2vec 和 GloVe（后者也利用了计数信息）。在此，我们将探索 GloVe 生成的嵌入。关于 word2vec 和 GloVe 算法的更多细节，请回顾课堂笔记和讲义幻灯片。如果您想挑战自己，可以尝试阅读 [GloVe 的原始论文](https://nlp.stanford.edu/pubs/glove.pdf)。

然后运行以下单元格，将 GloVe 向量加载到内存中。**注意**：如果您是首次运行这些单元格（即下载嵌入模型），将需要几分钟时间。如果您之前运行过这些单元格，重新运行时将直接加载模型而无需重新下载，大约需要 1 到 2 分钟。

In [ ]:
def load_embedding_model():
    """ 加载 GloVe 向量
        返回：
            wv_from_bin: 全部 400000 个嵌入，每个长度为 200
    """
    import gensim.downloader as api
    wv_from_bin = api.load("glove-wiki-gigaword-200")
    print("已加载词表大小 %i" % len(list(wv_from_bin.index_to_key)))
    return wv_from_bin
wv_from_bin = load_embedding_model()

#### 注意：如果您收到 "reset by peer"（连接被重置）错误，请重新运行该单元格以重新开始下载。

### 降低词嵌入的维度

让我们直接将 GloVe 嵌入与共现矩阵的嵌入进行比较。为避免内存不足，我们将使用 40000 个 GloVe 向量的子集。
运行以下单元格以：

1. 将 40000 个 GloVe 向量放入矩阵 M
2. 运行 `reduce_to_k_dim`（您的截断 SVD 函数），将向量从 200 维降维到 2 维。

In [ ]:
def get_matrix_of_vectors(wv_from_bin, required_words):
    """ 将 GloVe 向量放入矩阵 M。
        参数：
            wv_from_bin: KeyedVectors 对象；从文件加载的 400000 个 GloVe 向量
        返回：
            M: 形状为 (单词数, 200) 的 numpy 矩阵，包含向量
            word2ind: 将每个单词映射到 M 中行号的字典
    """
    import random
    words = list(wv_from_bin.index_to_key)
    print("正在打乱单词顺序...")
    random.seed(225)
    random.shuffle(words)
    print("正在将 %i 个词放入 word2ind 和矩阵 M..." % len(words))
    word2ind = {}
    M = []
    curInd = 0
    for w in words:
        try:
            M.append(wv_from_bin.get_vector(w))
            word2ind[w] = curInd
            curInd += 1
        except KeyError:
            continue
    for w in required_words:
        if w in words:
            continue
        try:
            M.append(wv_from_bin.get_vector(w))
            word2ind[w] = curInd
            curInd += 1
        except KeyError:
            continue
    M = np.stack(M)
    print("完成。")
    return M, word2ind

In [ ]:
# -----------------------------------------------------------------
# 运行此单元格，将 200 维词嵌入降维至 k 维
# 注意：此步骤应运行很快
# -----------------------------------------------------------------
M, word2ind = get_matrix_of_vectors(wv_from_bin, words)
M_reduced = reduce_to_k_dim(M, k=2)

# 重新缩放（归一化）各行，使每行具有单位长度
M_lengths = np.linalg.norm(M_reduced, axis=1)
M_reduced_normalized = M_reduced / M_lengths[:, np.newaxis] # 广播

**注意：如果您在本地计算机上遇到内存不足的问题，请尝试关闭其他应用程序以释放更多内存。您可以尝试重启计算机以释放额外内存，然后立即运行 Jupyter 笔记本，检查是否能正常加载词向量。如果在此之后仍然无法在本地计算机上加载嵌入，请前往答疑时间或联系课程助教。**

### 问题 2.1：GloVe 绘图分析 [文字题]（3分）

运行以下单元格，绘制 `['movie', 'book', 'mysterious', 'story', 'fascinating', 'good', 'interesting', 'large', 'massive', 'huge']` 的二维 GloVe 嵌入。

In [ ]:
words = ['movie', 'book', 'mysterious', 'story', 'fascinating', 'good', 'interesting', 'large', 'massive', 'huge']

plot_embeddings(M_reduced_normalized, word2ind, words)

**请验证您的图与作业压缩包中的 "question_2.1.png" 一致。如果不一致，请使用 "question_2.1.png" 中的图（以及适用的 "question_1.5.png" 中的图）来回答接下来的两个问题。**

a. 此图与之前由共现矩阵生成的图有何不同？又有何相似之处？

#### <font color="red">在此处填写您的答案。</font>

b. 为什么 GloVe 图（question_2.1.png）可能与之前由共现矩阵生成的图（question_1.5.png）不同？

#### <font color="red">在此处填写您的答案。</font>

### 余弦相似度

现在我们有了词向量，需要一种方法来根据这些向量量化单个词之间的相似度。余弦相似度就是这样一种度量方法。我们将用它来寻找彼此"接近"和"远离"的词。

我们可以将 n 维向量视为 n 维空间中的点。从这个角度来看，[L1](http://mathworld.wolfram.com/L1-Norm.html) 和 [L2](http://mathworld.wolfram.com/L2-Norm.html) 距离有助于量化在这两个点之间"需要移动"的空间距离。另一种方法是考察两个向量之间的夹角。根据三角学，我们知道：

<img src="./imgs/inner_product.png" width=20% style="float: center;"></img>

我们可以不计算实际角度，而是将相似度表示为 $similarity = cos(\Theta)$。正式地，两个向量 $p$ 和 $q$ 之间的[余弦相似度](https://en.wikipedia.org/wiki/Cosine_similarity) $s$ 定义为：

$$s = \frac{p \cdot q}{||p|| ||q||}, \textrm{ 其中 } s \in [-1, 1]$$

### 问题 2.2：多义词（1.5分）[编程 + 文字题]

多义词（polysemes）和同音异义词（homonyms）是指具有多个含义的词（参见此[维基百科页面](https://en.wikipedia.org/wiki/Polysemy)了解更多关于多义词和同音异义词之间区别的信息）。找到一个*至少有两个不同含义*的词，使得其按余弦相似度排名前 10 的最相似词中包含*两种*含义的相关词。例如，"leaves" 的前 10 个最相似词中既有 "go_away"（离开）的含义，又有 "a_structure_of_a_plant"（植物结构）的含义；"scoop" 则同时有 "handed_waffle_cone"（递蛋筒）和 "lowdown"（内幕消息）的含义。您可能需要尝试多个多义词或同音异义词才能找到一个符合条件的。

请说明您发现的词及其在前 10 个最相似词中出现的多个含义。您认为为什么您尝试的许多多义词或同音异义词没有成功（即前 10 个最相似词中只包含该词**其中一种**含义）？

**注意**：您应使用 `wv_from_bin.most_similar(word)` 函数获取前 10 个最相似词。该函数根据词汇表中所有其他词与给定词的余弦相似度进行排名。如需更多帮助，请查阅 __[GenSim 文档](https://radimrehurek.com/gensim/models/keyedvectors.html#gensim.models.keyedvectors.FastTextKeyedVectors.most_similar)__。

In [ ]:
# ------------------
# 在此编写您的实现代码。


# ------------------

#### <font color="red">在此处填写您的答案。</font>

### 问题 2.3：同义词与反义词（2分）[编程 + 文字题]

在讨论余弦相似度时，使用余弦距离通常更为方便，余弦距离即为 1 减去余弦相似度。

找到三个词 $(w_1,w_2,w_3)$，其中 $w_1$ 和 $w_2$ 是同义词，$w_1$ 和 $w_3$ 是反义词，但余弦距离 $(w_1,w_3) <$ 余弦距离 $(w_1,w_2)$。

例如，$w_1$="happy" 与 $w_3$="sad" 的距离比与 $w_2$="cheerful" 更近。请找一个满足上述条件的不同例子。找到例子后，请给出一个可能的解释，说明为什么会出现这种与直觉相反的结果。

此处应使用 `wv_from_bin.distance(w1, w2)` 函数来计算两个词之间的余弦距离。如需更多帮助，请参阅 __[GenSim 文档](https://radimrehurek.com/gensim/models/keyedvectors.html#gensim.models.keyedvectors.FastTextKeyedVectors.distance)__。

In [ ]:
# ------------------
# 在此编写您的实现代码。


# ------------------

#### <font color="red">在此处填写您的答案。</font>

### 问题 2.4：词向量类比 [文字题]（1.5分）

词向量已被证明*有时*具备解决类比问题的能力。

例如，对于类比 "man : grandfather :: woman : x"（读作：man 之于 grandfather 相当于 woman 之于 x），x 是什么？

在下面的单元格中，我们向您展示如何使用 __[GenSim 文档](https://radimrehurek.com/gensim/models/keyedvectors.html#gensim.models.keyedvectors.KeyedVectors.most_similar)__ 中的 `most_similar` 函数来找到 x。该函数查找与 `positive` 列表中的词最相似且与 `negative` 列表中的词最不相似的词（同时省略输入词本身，因为输入词往往是最相似的；参见[此论文](https://www.aclweb.org/anthology/N18-2039.pdf)）。类比的答案将具有最高的余弦相似度（返回的数值最大）。

In [ ]:
# 运行此单元格来回答类比问题 -- man : grandfather :: woman : x
pprint.pprint(wv_from_bin.most_similar(positive=['woman', 'grandfather'], negative=['man']))

设 $m$、$g$、$w$ 和 $x$ 分别表示 `man`、`grandfather`、`woman` 和答案的词向量。**仅**使用向量 $m$、$g$、$w$ 以及向量算术运算符 $+$ 和 $-$，给出使与 $x$ 的余弦相似度最大化的表达式。

提示：回顾词向量只是表示单词的多维向量。画一个二维示例，使用各向量的任意位置可能会有所帮助。`man` 和 `woman` 在坐标平面中相对于 `grandfather` 和答案的位置关系如何？

#### <font color="red">在此处填写您的答案。</font>

### 问题 2.5：寻找类比 [编程 + 文字题]（1.5分）

a. 对于上面的例子，"grandmother"（祖母）显然完成了该类比。但请给出一个直观的解释，说明为什么 `most_similar` 函数会给出 "granddaughter"（孙女）、"daughter"（女儿）或 "mother"（母亲）这样的词？

#### <font color="red">在此处填写您的答案。</font>

b. 找到一个根据这些向量成立的类比示例（即目标词排名第一）。在您的解答中，请以 x:y :: a:b 的形式完整陈述该类比。如果您认为该类比较为复杂，请用一两句话解释为什么该类比成立。

**注意**：您可能需要尝试许多类比才能找到一个有效的！

In [ ]:
# 例如: x, y, a, b = ("", "", "", "")
# ------------------
# 在此编写您的实现代码。


# ------------------

# 测试答案
assert wv_from_bin.most_similar(positive=[a, y], negative=[x])[0][0] == b

#### <font color="red">在此处填写您的答案。</font>

### 问题 2.6：不正确的类比 [编程 + 文字题]（1.5分）

a. 在下方，我们期望看到类比 "hand : glove :: foot : **sock**"（手之于手套相当于脚之于袜子），但结果却出乎意料。请给出一个可能的原因，解释为什么这个特定的类比会得出这样的结果？

In [ ]:
pprint.pprint(wv_from_bin.most_similar(positive=['foot', 'glove'], negative=['hand']))

#### <font color="red">在此处填写您的答案。</font>

b. 找到另一个根据这些向量*不*成立的类比示例。在您的解答中，以 x:y :: a:b 的形式陈述预期的类比，并陈述根据词向量得到的**错误**的 b 值（在上面的例子中，这将是 **'45,000-square'**）。

In [ ]:
# 例如: x, y, a, b = ("", "", "", "")
# ------------------
# 在此编写您的实现代码。


# ------------------
pprint.pprint(wv_from_bin.most_similar(positive=[a, y], negative=[x]))
assert wv_from_bin.most_similar(positive=[a, y], negative=[x])[0][0] != b

#### <font color="red">在此处填写您的答案。</font>

### 问题 2.7：词向量中偏见的引导分析 [文字题]（1分）

重要的是要认识到我们的词嵌入中隐含的偏见（性别、种族、性取向等）。偏见可能具有危险性，因为使用这些模型的应用程序可能会通过它们来强化刻板印象。

运行下方的单元格，考察 (a) 哪些词与 "man"（男人）和 "profession"（职业）最相似且与 "woman"（女人）最不相似，以及 (b) 哪些词与 "woman" 和 "profession" 最相似且与 "man" 最不相似。指出女性关联词列表和男性关联词列表之间的差异，并解释它如何反映性别偏见。

In [ ]:
# 运行此单元格
# 这里 `positive` 表示要相似的词列表，`negative` 表示要最不相似的词列表。

pprint.pprint(wv_from_bin.most_similar(positive=['man', 'profession'], negative=['woman']))
print()
pprint.pprint(wv_from_bin.most_similar(positive=['woman', 'profession'], negative=['man']))

#### <font color="red">在此处填写您的答案。</font>

### 问题 2.8：词向量中偏见的独立分析 [编程 + 文字题]（1分）

使用 `most_similar` 函数找到另一对类比，展示这些向量所表现出的某种偏见。请简要解释您发现的偏见示例。

In [ ]:
# ------------------
# 在此编写您的实现代码。


# ------------------

#### <font color="red">在此处填写您的答案。</font>

### 问题 2.9：关于偏见的思考 [文字题]（2分）

a. 给出一种解释，说明偏见是如何进入词向量的。简要描述一个展示这种偏见来源的真实案例。您的真实案例应聚焦于词向量，而非其他 AI 系统（如 ChatGPT）中的偏见。

#### <font color="red">在此处填写您的答案。</font>

b. 有什么方法可以缓解词向量所表现出的偏见？简要描述一个展示此方法的真实案例。


#### <font color="red">在此处填写您的答案。</font>

# <font color="blue"> 提交说明</font>

1. 点击 Jupyter 笔记本顶部的保存按钮。
2. 选择 Cell -> All Output -> Clear。这将清除所有单元格的输出（但会保留所有单元格的内容）。
2. 选择 Cell -> Run All。这将按顺序运行所有单元格，需要几分钟时间。
3. 重新运行所有单元格后，选择 File -> Download as -> PDF via LaTeX（如果使用 "PDF via LaTeX" 遇到问题，您也可以将网页保存为 PDF。<font color='blue'>请确保您的所有解答，尤其是代码部分，都显示在 PDF 中</font>，代码单元格中因行未换行而被截断的提供代码可以接受）。
4. 查看 PDF 文件，确保您的所有解答都在其中且显示正确。评分者只会查看 PDF！
5. 在 Gradescope 上提交您的 PDF。